Step 1: GPU Availability Check and Ultralytics Installation

In [ ]:

!nvidia-smi
!pip install -q ultralytics


Step 2: Mount Google Drive and Locate the Dataset

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Step 3: Locate the CivicResolve Dataset Configuration

---



In [ ]:

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
matches = list(drive_root.rglob("04_Final_YOLO_Dataset/data.yaml"))

print("Matching dataset configurations:", len(matches))

for path in matches:
    print(path)


Step 4: Verify Dataset Structure and Configuration Files

In [ ]:

from pathlib import Path
import yaml

dataset_dir = Path(
    "/content/drive/MyDrive/CivicResolve_Datasets/04_Final_YOLO_Dataset"
)
yaml_path = dataset_dir / "data.yaml"

# Check dataset folders
for split in ["train", "val", "test"]:
    image_dir = dataset_dir / "images" / split
    label_dir = dataset_dir / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]
    labels = list(label_dir.glob("*.txt"))

    print(f"{split}: {len(images)} images, {len(labels)} label files")

# Fix dataset paths
config = yaml.safe_load(yaml_path.read_text())

config["path"] = str(dataset_dir)
config["train"] = "images/train"
config["val"] = "images/val"
config["test"] = "images/test"

yaml_path.write_text(yaml.safe_dump(config, sort_keys=False))

print("\nUpdated data.yaml:")
print(yaml_path.read_text())


Step 5: Verify Dataset Images and Label Files

In [ ]:

from pathlib import Path
import time

dataset_dir = Path(
    "/content/drive/MyDrive/CivicResolve_Datasets/04_Final_YOLO_Dataset"
)

for split in ["train", "val", "test"]:
    folder = dataset_dir / "images" / split
    start = time.time()
    count = sum(
        1 for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in
        [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    )
    print(f"{split}: {count} images | {time.time()-start:.1f} seconds")


Step 6: Copy Dataset to Colab Local Storage

In [ ]:

import shutil
from pathlib import Path

source = Path(
    "/content/drive/MyDrive/CivicResolve_Datasets/04_Final_YOLO_Dataset"
)
destination = Path("/content/CivicResolve_Dataset")

if destination.exists():
    shutil.rmtree(destination)

print("Copying dataset to Colab local storage...")
shutil.copytree(source, destination)

print("Copy complete:", destination.exists())
print("Dataset YAML exists:", (destination / "data.yaml").exists())


Step 7: Update the YOLO Dataset Configuration (data.yaml)

In [ ]:

import yaml

yaml_path = destination / "data.yaml"
config = yaml.safe_load(yaml_path.read_text())

config["path"] = str(destination)
config["train"] = "images/train"
config["val"] = "images/val"
config["test"] = "images/test"

yaml_path.write_text(yaml.safe_dump(config, sort_keys=False))

for split in ["train", "val", "test"]:
    folder = destination / "images" / split
    print(split, "images:", len(list(folder.iterdir())))

print("\nUpdated YAML:")
print(yaml_path.read_text())


Step 8: Train the YOLOv8n Model

In [ ]:

from ultralytics import YOLO
import torch

print("GPU available:", torch.cuda.is_available())
print("Dataset:", yaml_path)

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=10,
    imgsz=416,
    batch=8,
    workers=2,
    cache=False,
    device=0,
    project="/content/civicresolve_runs",
    name="civicresolve_trial2",
    seed=42,
    plots=True
)


Step 9: Review Training Results and Performance Metrics

In [ ]:

from pathlib import Path
import shutil

best_path = Path(
    "/content/civicresolve_runs/civicresolve_trial2/weights/best.pt"
)

destination = Path(
    "/content/drive/MyDrive/CivicResolve_best.pt"
)

if best_path.exists():
    shutil.copy2(best_path, destination)
    print("SUCCESS: Model saved to Google Drive!")
    print("Location:", destination)
    print("Size (MB):", round(destination.stat().st_size / (1024 * 1024), 2))
else:
    print("best.pt not found. Searching for saved model files...")
    for path in Path("/content/civicresolve_runs").rglob("best.pt"):
        print(path)


Step 10: Load the Trained Model

In [ ]:

from ultralytics import YOLO

model_path = "/content/drive/MyDrive/CivicResolve_best.pt"
trained_model = YOLO(model_path)

print("Model loaded successfully!")
print("Detected classes:", trained_model.names)


Step 11: Test the Model on a Sample Civic-Issue Image

In [ ]:

from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded.keys()))

print("Uploaded image:", image_path)


Step 12: Locate the Trained Model Weights (best.pt)

In [ ]:

from ultralytics import YOLO
from pathlib import Path
from IPython.display import display, Image

model_path = "/content/drive/MyDrive/CivicResolve_best.pt"

if not Path(model_path).exists():
    raise FileNotFoundError(
        "Model not found. Complete Step 9 to save best.pt first."
    )

model = YOLO(model_path)

results = model.predict(
    source=image_path,
    imgsz=416,
    conf=0.25,
    save=True
)

print("Detected classes:", model.names)

for result in results:
    print("Boxes detected:", len(result.boxes))
    output_path = Path(result.save_dir) / Path(image_path).name
    if output_path.exists():
        display(Image(filename=str(output_path)))


Step 13: Verify the Trained Model File

In [ ]:

from pathlib import Path
import shutil

prediction_dir = Path("/content/runs/detect/predict")
drive_output = Path("/content/drive/MyDrive/CivicResolve_Predictions")

drive_output.mkdir(parents=True, exist_ok=True)

images = [
    p for p in prediction_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

if images:
    for image in images:
        destination = drive_output / image.name
        shutil.copy2(image, destination)
        print("Saved:", destination)
else:
    print("No images found. Check the prediction output directory.")


Step 14: Save the Trained Model to Google Drive

In [ ]:

from pathlib import Path

model_path = Path("/content/drive/MyDrive/CivicResolve_best.pt")

if model_path.exists():
    print("✅ Model found!")
    print("Path:", model_path)
    print("Size:", round(model_path.stat().st_size / (1024 * 1024), 2), "MB")
else:
    print("❌ Model not found. Searching for best.pt...")
    for p in Path("/content/civicresolve_runs").rglob("best.pt"):
        print(p)


Step 15 — Download the Trained Model

In [ ]:

from google.colab import files

files.download("/content/drive/MyDrive/CivicResolve_best.pt")


Step 16 — Move the Model into Your VS Code Project

In [ ]:
from pathlib import Path

for p in Path("/content").rglob("best.pt"):
    print(p)

Step 17 — Update the Model Path

In [ ]:
import shutil
from pathlib import Path

source = Path(
    "/content/civicresolve_runs/civicresolve_trial2/weights/best.pt"
)

destination = Path(
    "/content/drive/MyDrive/CivicResolve_best.pt"
)

if source.exists():
    shutil.copy2(source, destination)
    print("Model saved to Google Drive:", destination)
else:
    print("Source not found. Use the path printed in Step 1.")